In [ ]:
from google.colab import drive
drive.mount("/content/drive")
BASE_PATH = "/content/drive/MyDrive/Autojudge ACM Project"


Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import mean_absolute_error, mean_squared_error

from scipy.sparse import hstack



In [ ]:
df = pd.read_json(f"{BASE_PATH}/data.json", lines=True)
df.head()

,title,description,input_description,output_description,sample_io,problem_class,problem_score,url
0,Uuu,Unununium (Uuu) was the name of the chemical\n...,The input consists of one line with two intege...,The output consists of $M$ lines where the $i$...,"[{'input': '7 10', 'output': '1 2 2 3 1 3 3 4 ...",hard,9.7,https://open.kattis.com/problems/uuu
1,House Building,A number of eccentrics from central New York h...,"The input consists of $10$ test cases, which a...",Print $K$ lines with\n the positions of the...,"[{'input': '0 2 3 2 50 60 50 30 50 40', 'outpu...",hard,9.7,https://open.kattis.com/problems/husbygge
2,Mario or Luigi,Mario and Luigi are playing a game where they ...,,,"[{'input': '', 'output': ''}]",hard,9.6,https://open.kattis.com/problems/marioorluigi
3,The Wire Ghost,Žofka is bending a copper wire. She starts wit...,The first line contains two integers $L$ and $...,The output consists of a single line consistin...,"[{'input': '4 3 3 C 2 C 1 C', 'output': 'GHOST...",hard,9.6,https://open.kattis.com/problems/thewireghost
4,Barking Up The Wrong Tree,"Your dog Spot is let loose in the park. Well, ...",The first line of input consists of two intege...,Write a single line containing the length need...,"[{'input': '2 0 10 0 10 10', 'output': '14.14'...",hard,9.6,https://open.kattis.com/problems/barktree


In [ ]:
def normalize_text(x):
    if isinstance(x, list):
        return " ".join(map(str, x))
    if isinstance(x, dict):
        return " ".join([f"{k} {v}" for k, v in x.items()])
    if pd.isna(x):
        return ""
    return str(x)

text_cols = [
    "title",
    "description",
    "input_description",
    "output_description",
    "sample_io"
]

for col in text_cols:
    df[col] = df[col].apply(normalize_text)

df["full_text"] = df[text_cols].agg(" ".join, axis=1)
df["full_text"] = df["full_text"].str.lower()


In [ ]:
class_map = {"easy": 0, "medium": 1, "hard": 2}
df["problem_class_encoded"] = df["problem_class"].str.lower().map(class_map)

df["problem_score"] = pd.to_numeric(df["problem_score"], errors="coerce")

df = df.dropna(subset=["problem_class_encoded", "problem_score"])


In [ ]:
KEYWORDS = [
    "graph", "tree", "dp", "dynamic programming",
    "recursion", "greedy", "binary search",
    "matrix", "modulo", "shortest path",
    "dfs", "bfs", "segment tree"
]

class HandcraftedFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data = pd.DataFrame()
        data["text_length"] = X.apply(len)
        data["math_symbol_count"] = X.apply(
            lambda x: len(re.findall(r"[+\-*/%=<>^]", x))
        )

        for kw in KEYWORDS:
            data[f"kw_{kw.replace(' ', '_')}"] = X.str.count(kw)

        return data.values


In [ ]:
class FeatureUnion(BaseEstimator, TransformerMixin):
    def __init__(self, tfidf, handcrafted):
        self.tfidf = tfidf
        self.handcrafted = handcrafted

    def fit(self, X, y=None):
        self.tfidf.fit(X)
        self.handcrafted.fit(X)
        return self

    def transform(self, X):
        X_tfidf = self.tfidf.transform(X)
        X_hand = self.handcrafted.transform(X)
        return hstack([X_tfidf, X_hand])


In [ ]:
%%writefile custom_transformers.py
import pandas as pd
import re
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import hstack

KEYWORDS = [
    "graph", "tree", "dp", "dynamic programming",
    "recursion", "greedy", "binary search",
    "matrix", "modulo", "shortest path",
    "dfs", "bfs", "segment tree"
]

class HandcraftedFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        data = pd.DataFrame()
        data["text_length"] = X.apply(len)
        data["math_symbol_count"] = X.apply(
            lambda x: len(re.findall(r"[+\-*/%=<>^]", x))
        )

        for kw in KEYWORDS:
            data[f"kw_{kw.replace(' ', '_')}"] = X.str.count(kw)

        return data.values


class FeatureUnion(BaseEstimator, TransformerMixin):
    def __init__(self, tfidf, handcrafted):
        self.tfidf = tfidf
        self.handcrafted = handcrafted

    def fit(self, X, y=None):
        self.tfidf.fit(X)
        self.handcrafted.fit(X)
        return self

    def transform(self, X):
        X1 = self.tfidf.transform(X)
        X2 = self.handcrafted.transform(X)
        return hstack([X1, X2])


Writing custom_transformers.py


In [ ]:
from custom_transformers import HandcraftedFeatures, FeatureUnion


In [ ]:
clf_pipeline = Pipeline([
    ("features", FeatureUnion(
        tfidf=TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            stop_words="english",
            min_df=3
        ),
        handcrafted=HandcraftedFeatures()
    )),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])


In [ ]:
reg_pipeline = Pipeline([
    ("features", FeatureUnion(
        tfidf=TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            stop_words="english",
            min_df=3
        ),
        handcrafted=HandcraftedFeatures()
    )),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])


In [ ]:
X = df["full_text"]
y_class = df["problem_class_encoded"]
y_reg = df["problem_score"]

X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test = train_test_split(
    X, y_class, y_reg,
    test_size=0.2,
    random_state=42,
    stratify=y_class
)


In [ ]:
clf_pipeline.fit(X_train, y_class_train)
reg_pipeline.fit(X_train, y_reg_train)


Pipeline(steps=[('features',
                 FeatureUnion(handcrafted=HandcraftedFeatures(),
                              tfidf=TfidfVectorizer(max_features=5000, min_df=3,
                                                    ngram_range=(1, 2),
                                                    stop_words='english'))),
                ('model',
                 RandomForestRegressor(n_estimators=200, n_jobs=-1,
                                       random_state=42))])

In [ ]:
y_pred_class = clf_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_class_test, y_pred_class))
print(classification_report(y_class_test, y_pred_class, target_names=["Easy", "Medium", "Hard"]))


Accuracy: 0.528554070473876
              precision    recall  f1-score   support

        Easy       0.70      0.26      0.38       153
      Medium       0.40      0.20      0.27       281
        Hard       0.54      0.87      0.67       389

    accuracy                           0.53       823
   macro avg       0.55      0.44      0.44       823
weighted avg       0.52      0.53      0.48       823



In [ ]:
y_pred_reg = reg_pipeline.predict(X_test)

print("MAE:", mean_absolute_error(y_reg_test, y_pred_reg))
print("RMSE:", np.sqrt(mean_squared_error(y_reg_test, y_pred_reg)))


MAE: 1.6038736330498178
RMSE: 1.9219470600988469


In [ ]:
clf_pipeline.fit(X, y_class)
reg_pipeline.fit(X, y_reg)


Pipeline(steps=[('features',
                 FeatureUnion(handcrafted=HandcraftedFeatures(),
                              tfidf=TfidfVectorizer(max_features=5000, min_df=3,
                                                    ngram_range=(1, 2),
                                                    stop_words='english'))),
                ('model',
                 RandomForestRegressor(n_estimators=200, n_jobs=-1,
                                       random_state=42))])

In [ ]:
joblib.dump(clf_pipeline, "difficulty_classifier.pkl")
joblib.dump(reg_pipeline, "difficulty_regressor.pkl")

print(" Pipelines saved successfully")


 Pipelines saved successfully
